In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

### Batching

In [ ]:
records, _, _ = graph.driver.execute_query("""
MATCH (o:Officer)
RETURN id(o) AS id
ORDER BY id(o)
""")

ids = [r["id"] for r in records]

print(len(ids))

In [ ]:
def chunk(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

batches = list(chunk(ids, 1000))

In [14]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env_panama.json")

In [ ]:
rules = []

batch_size = 1000

for i in range(0, len(ids), batch_size):
    batch_ids = ids[i:i+batch_size]
    ids_str = ",".join(map(str, batch_ids))

    rules.append(Rule(
        f"""
        MATCH (o)
        WHERE id(o) IN [{ids_str}]
        AND o:Officer AND NOT o:Person
        GENERATE
        ((o):Person {{
            name = o.name
        }})
        """,
        env=env,
        type_strict=False,
    ))

my_transform = Transformation(rules)
my_transform.apply_on(graph)

In [15]:
Rule1 = Rule('''
MATCH (i:Intermediary)-[:intermediary_of]->(e:Entity)
WITH i, e LIMIT 2000
GENERATE
(c = (i.internal_id):T_country {
    ccode = i.country_codes,
    name = i.name,
    internal_id = toInteger(i.internal_id)
})<-[():LOCATED_IN]-(x = (i):T_director {
    name = i.name
})-[():DIRECTOR_OF]->(y = (e):)
''', env=env, type_strict=True)


Rule2 = Rule('''
MATCH (o1:Officer)-[:director_of]->(o2:Officer)-[:director_of*]->()
WITH o1, o2 LIMIT 500
GENERATE
(c = (o1.sourceID):T_country {
    valid = o1.valid_until,
    name = o1.name
})<-[():LOCATED_IN]-(x = (o1):T_director {
    name = o1.name
})
''', env=env, type_strict=True)


Rule3 = Rule('''
MATCH (o1:Officer)-[:officer_of*]->(e:Entity)-[:registered_address]->(a:Address)
WITH o1, e, a LIMIT 2000
GENERATE
(c = (a):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_director)
''', env=env, type_strict=True)

Rule4 = Rule('''
MATCH (o1:Intermediary)-[:intermediary_of]->(e:Entity)-[:registered_address]->(a:Address)
WITH o1, e, a LIMIT 2000
GENERATE
(c = (a):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_intermediary)
''', env=env, type_strict=True)

Rule5 = Rule('''
MATCH (o1:Entity)-[:officer_of*]->(a:Address)
GENERATE
(c = (a.countryname):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_entity)
''', env=env, type_strict=True)

In [16]:
my_transform = Transformation([Rule1])
my_transform.apply_on(graph)

Index: Added 1 index, completed after 62 ms.

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (i:Intermediary)-[:intermediary_of]->(e:Entity)\nWITH i, e LIMIT 2000', 'constructors': [{'src': {'alias': 'x', 'ids': ['i'], 'labels': ['T_director'], 'properties': [{'key': 'name', 'value': 'i.name\n'}]}, 'edge': {'ids': [], 'labels': ['LOCATED_IN']}, 'tgt': {'alias': 'c', 'ids': ['i.internal_id'], 'labels': ['T_country'], 'properties': [{'key': 'ccode', 'value': 'i.country_codes'}, {'key': 'name', 'value': 'i.name'}, {'key': 'internal_id', 'value': 'toInteger(i.internal_id)\n'}]}}, {'src': {'alias': 'x'}, 'edge': {'ids': [], 'labels': ['DIRECTOR_OF']}, 'tgt': {'alias': 'y', 'ids': ['e']}}]}
AST:
PropertyAccess
    ├── var: i
    └── prop: name
AST:
PropertyAccess
    ├── var: i
    └── prop: country_codes
AST:
PropertyAccess
    ├── var: i
    └── prop: name
AST:
FunctionCall: toInteger
    └── PropertyAccess
        ├── var: i
        └── prop: internal_id
Rule: Added 2208 labels, 

1937

### Abort Transformation

In [17]:
my_transform.abort()

Index: Removed 1 index, completed after 11 ms.
Abort: Deleted 2104 nodes, deleted 2052 relationships, completed after 117 ms.
